# Ground-truth pace — interactive tuner

Reconstruct a clean per-second pace from Garmin running dynamics and **tune the estimator live** with sliders, using the `pace_predict` library.

Two reconstructions are compared:

- **Batch smoother** (default, `SmootherParams`): one global fit that models the GNSS lag, keeps speed smooth except inside cadence-detected transitions, ignores GNSS faults, and holds the ≥1 km laps exactly.
- **Segment reconstruction** (`method="segments"`, `PhaseParams` + `SegmentParams`): segments the run by cadence phase, holds each steady segment near-constant, ramps the transitions, and calibrates to the ≥1 km laps.

Every threshold is exposed as a slider so you can fit it per runner.

> **Privacy.** This reads FIT files from `data/` (gitignored) and shows home GPS-derived data, so **run it locally**. Outputs are stripped before commit (nbstripout) — no GPS/health data is ever committed. CI verifies committed notebooks carry no outputs.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

from pace_predict.groundtruth import (
    ACCEL, DECEL, STEADY, PhaseParams, SegmentParams,
    ground_truth_speed, integrate_speed, median_dt_s, phase_labels, phase_segments,
)
from pace_predict.inventory import build_inventory
from pace_predict.io import load_activity
from pace_predict.io import schema as S


def _repo_root() -> Path:
    """Repo root (dir with pyproject.toml), so the notebook works from any kernel cwd."""
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / 'pyproject.toml').exists():
            return candidate
    return Path.cwd()


DATA = _repo_root() / 'data' / 'fit'


def to_pace(speed_mps):
    """m/s -> min/km, blanking near-stops so pauses don't dominate the axis."""
    v = np.asarray(speed_mps, float)
    v = np.where(v > 0.6, v, np.nan)
    return 1000.0 / v / 60.0

## Dataset inventory
One row per activity. `track` flags runs on a 400 m oval (GNSS overshoots — held out of calibration); `dyn_pace` marks the embedded `dynamicsPace` field, the baseline to beat.

In [ ]:
inventory = build_inventory(sorted(DATA.glob('*.fit')))
if inventory.empty:
    raise RuntimeError(f'No activities under {DATA} — add Garmin .fit files (see data/README.md).')
inventory[['file', 'sport', 'dur_min', 'dist_km', 'laps', 'track', 'cadence_spm', 'gct_ms', 'dyn_pace']]

## Segment reconstruction tuner

Pick a run and drag the sliders; the ground-truth line (blue) and its accel/decel markers update live. Grey relief along the bottom is the elevation profile; faint vertical lines are lap boundaries.

**Phase classification** — `thr_spm_s` (cadence-slope threshold), `smooth_s` (cadence smoothing), `min_phase_s` (shortest transition kept), `min_delta_spm` (smallest plateau-level change kept). **Reconstruction** — `max_drift_mps` (how much a steady segment may slope for a hill), `prefilter_s` (spike-reject median). **Dropouts** — `dropout_gap_s` (longest GNSS dropout interpolated).

What to look for: steady segments should sit flat at the right level, transitions should be crisp at each rep, and the calibration error should stay small (< ~1%).

In [ ]:
def compute(run, params):
    act = load_activity(DATA / run)
    r = act.records
    dt = median_dt_s(r)
    minutes = r[S.T_S].to_numpy() / 60.0
    gt = ground_truth_speed(act, method='segments', params=params)
    labels = phase_labels(r[S.CADENCE_SPM].to_numpy(), dt, params=params.phase)
    alt = r[S.ALTITUDE_M].to_numpy()
    if np.isfinite(alt).any():
        lo, hi = np.nanmin(alt), np.nanmax(alt)
        alt_norm = (alt - lo) / (hi - lo) if hi > lo else np.zeros_like(alt)
    else:
        alt_norm = np.zeros_like(alt)
    start0 = r[S.TIMESTAMP].min()
    lap_min = [(e - start0).total_seconds() / 60.0 for e in act.laps[S.LAP_END_TIME].dropna()]

    dist_gt = float(integrate_speed(gt, dt)[-1])
    dist_rec = float(np.nanmax(r[S.DISTANCE_M].to_numpy()))
    has_dist = bool(np.isfinite(dist_rec) and dist_rec > 0)
    cal_pct = 100.0 * (dist_gt - dist_rec) / dist_rec if has_dist else float('nan')
    seg = phase_segments(labels)
    counts = {p: sum(1 for s in seg if s.phase == p) for p in (ACCEL, STEADY, DECEL)}
    steady_pct = 100.0 * float(np.mean(labels == STEADY))
    return {
        'minutes': minutes, 'raw_pace': to_pace(r[S.SPEED_MPS].to_numpy()),
        'gt_pace': to_pace(gt), 'labels': labels, 'alt_norm': alt_norm, 'lap_min': lap_min,
        'cal_pct': cal_pct, 'counts': counts, 'steady_pct': steady_pct,
    }


def _params_from(sliders):
    return SegmentParams(
        phase=PhaseParams(
            thr_spm_s=sliders['thr'].value, smooth_s=sliders['smooth'].value,
            min_phase_s=sliders['min_phase'].value, min_delta_spm=sliders['min_delta'].value,
        ),
        prefilter_s=sliders['prefilter'].value, max_drift_mps=sliders['max_drift'].value,
        dropout_gap_s=sliders['dropout'].value,
    )


def _lap_shapes(lap_min):
    return [dict(type='line', x0=x, x1=x, y0=0, y1=1, yref='paper',
                 line=dict(color='rgba(0,0,0,0.12)', width=1)) for x in lap_min]


# Widgets: run picker + one slider per tunable parameter.
runs = sorted(p.name for p in DATA.glob('*.fit'))
default_run = inventory.sort_values('laps', ascending=False)['file'].iloc[0]
run_dd = widgets.Dropdown(options=runs, value=default_run, description='run')
S_ = dict(style={'description_width': '110px'}, layout=widgets.Layout(width='320px'),
          continuous_update=False)
sliders = {
    'thr': widgets.FloatSlider(0.45, min=0.15, max=1.0, step=0.05, description='thr_spm_s', **S_),
    'smooth': widgets.FloatSlider(9, min=3, max=21, step=2, description='smooth_s', **S_),
    'min_phase': widgets.FloatSlider(3, min=1, max=10, step=1, description='min_phase_s', **S_),
    'min_delta': widgets.FloatSlider(2, min=0.5, max=6, step=0.5, description='min_delta_spm', **S_),
    'max_drift': widgets.FloatSlider(0.25, min=0.0, max=1.0, step=0.05, description='max_drift_mps', **S_),
    'prefilter': widgets.FloatSlider(7, min=1, max=21, step=2, description='prefilter_s', **S_),
    'dropout': widgets.FloatSlider(8, min=0, max=20, step=1, description='dropout_gap_s', **S_),
}
metrics = widgets.HTML()

fig = go.FigureWidget()
fig.add_trace(go.Scatter(yaxis='y2', fill='tozeroy', mode='none',
                         fillcolor='rgba(150,150,150,0.22)', hoverinfo='skip', showlegend=False))
fig.add_trace(go.Scatter(name='raw enhanced_speed', line=dict(color='lightgray', width=1)))
fig.add_trace(go.Scatter(name='ground truth', line=dict(color='royalblue', width=2.4)))
fig.add_trace(go.Scatter(name='accel', mode='markers', marker=dict(color='crimson', size=5)))
fig.add_trace(go.Scatter(name='decel', mode='markers', marker=dict(color='seagreen', size=5)))
fig.update_layout(height=520, margin=dict(t=40, b=40), xaxis=dict(title='time (min)'),
                  yaxis=dict(title='pace (min/km)', autorange='reversed'),
                  yaxis2=dict(overlaying='y', side='right', range=[0, 4], showgrid=False,
                              showticklabels=False, fixedrange=True),
                  legend=dict(orientation='h', y=1.08))


def update(_=None):
    d = compute(run_dd.value, _params_from(sliders))
    m, lab, gp = d['minutes'], d['labels'], d['gt_pace']
    with fig.batch_update():
        fig.data[0].x, fig.data[0].y = m, d['alt_norm']
        fig.data[1].x, fig.data[1].y = m, d['raw_pace']
        fig.data[2].x, fig.data[2].y = m, gp
        fig.data[3].x, fig.data[3].y = m[lab == ACCEL], gp[lab == ACCEL]
        fig.data[4].x, fig.data[4].y = m[lab == DECEL], gp[lab == DECEL]
        fig.layout.shapes = _lap_shapes(d['lap_min'])
        fig.layout.title.text = run_dd.value
        # Focus the pace axis on the running range (ignore the standing-start spike).
        finite = np.concatenate([d['raw_pace'], gp])
        finite = finite[np.isfinite(finite)]
        if finite.size:
            lo, hi = np.nanpercentile(finite, [2, 98])
            pad = 0.15 * (hi - lo) + 0.1
            # autorange must be off, else the initial autorange='reversed' ignores the range
            fig.layout.yaxis.update(range=[hi + pad, lo - pad], autorange=False)
    c = d['counts']
    metrics.value = (
        f"<b>calibration</b> {d['cal_pct']:+.2f}% vs recorded distance &nbsp;|&nbsp; "
        f"<b>steady</b> {d['steady_pct']:.0f}% of samples &nbsp;|&nbsp; "
        f"<b>segments</b> {c[STEADY]} steady / {c[ACCEL]} accel / {c[DECEL]} decel"
    )


run_dd.observe(update, names='value')
for s in sliders.values():
    s.observe(update, names='value')
update()

controls = widgets.VBox([run_dd, *sliders.values()])
display(widgets.VBox([metrics, widgets.HBox([controls, fig])]))

## Batch smoother tuner

The default reconstruction. Blue is the batch smoother, orange the segment reconstruction at its defaults for comparison.

**Lag** — `estimate lag per run` derives the GNSS lag time constant from how far `enhanced_speed` trails cadence changes; untick it to set `lag_s` by hand (0 = trust GNSS timing). **Shape** — `smoothing` (penalty strength; higher = calmer steady stretches), `order` (3 = piecewise-quadratic speed). **Transitions** — `transition_pad_s` widens each cadence-detected transition, `transition_floor` is the penalty left inside it (lower = crisper changes). **Robustness** — `outlier_scale` (m/s): GNSS samples much further than this from the fit lose influence.

What to look for: crisp rep starts without dips from GNSS faults, calm but not ruler-straight steady stretches, calibration within ~1%.

In [ ]:
from pace_predict.groundtruth import SmootherParams, estimate_gnss_lag_s, guard_dropouts

b_run = widgets.Dropdown(options=runs, value=default_run, description='run', **S_)
b_auto = widgets.Checkbox(value=True, description='estimate lag per run')
b_sliders = {
    'lag': widgets.FloatSlider(4, min=0, max=15, step=0.5, description='lag_s', **S_),
    'smoothing': widgets.FloatLogSlider(250, base=10, min=0, max=4, step=0.1, description='smoothing', **S_),
    'order': widgets.IntSlider(3, min=2, max=4, description='order', **S_),
    'pad': widgets.IntSlider(3, min=0, max=15, description='transition_pad_s', **S_),
    'floor': widgets.FloatLogSlider(0.01, base=10, min=-3, max=0, step=0.25, description='transition_floor', **S_),
    'outlier': widgets.FloatSlider(0.25, min=0.05, max=2.0, step=0.05, description='outlier_scale', **S_),
}
b_metrics = widgets.HTML()

bfig = go.FigureWidget()
bfig.add_trace(go.Scatter(yaxis='y2', fill='tozeroy', mode='none',
                          fillcolor='rgba(150,150,150,0.22)', hoverinfo='skip', showlegend=False))
bfig.add_trace(go.Scatter(name='raw enhanced_speed', line=dict(color='lightgray', width=1)))
bfig.add_trace(go.Scatter(name='segment reconstruction', line=dict(color='darkorange', width=1.3)))
bfig.add_trace(go.Scatter(name='batch smoother', line=dict(color='royalblue', width=2.4)))
bfig.update_layout(height=520, margin=dict(t=40, b=40), xaxis=dict(title='time (min)'),
                   yaxis=dict(title='pace (min/km)'),
                   yaxis2=dict(overlaying='y', side='right', range=[0, 4], showgrid=False,
                               showticklabels=False, fixedrange=True),
                   legend=dict(orientation='h', y=1.08))


_b_cache = {}  # run -> (activity, segment reference); batch sliders don't affect either


def b_update(_=None):
    if b_run.value not in _b_cache:
        _b_cache[b_run.value] = (load_activity(DATA / b_run.value), compute(b_run.value, SegmentParams()))
    act, d = _b_cache[b_run.value]
    r = act.records
    dt = median_dt_s(r)
    speed = guard_dropouts(np.nan_to_num(r[S.SPEED_MPS].to_numpy(), nan=0.0), dt)
    lag = (estimate_gnss_lag_s(speed, r[S.CADENCE_SPM].to_numpy(), dt)
           if b_auto.value else b_sliders['lag'].value)
    b_sliders['lag'].disabled = b_auto.value
    smoother = SmootherParams(
        lag_s=lag, smoothing=b_sliders['smoothing'].value, order=b_sliders['order'].value,
        transition_pad_s=b_sliders['pad'].value, transition_floor=b_sliders['floor'].value,
        outlier_scale_mps=b_sliders['outlier'].value,
    )
    batch = ground_truth_speed(act, smoother=smoother)
    m = d['minutes']
    with bfig.batch_update():
        bfig.data[0].x, bfig.data[0].y = m, d['alt_norm']
        bfig.data[1].x, bfig.data[1].y = m, d['raw_pace']
        bfig.data[2].x, bfig.data[2].y = m, d['gt_pace']
        bfig.data[3].x, bfig.data[3].y = m, to_pace(batch)
        bfig.layout.shapes = _lap_shapes(d['lap_min'])
        bfig.layout.title.text = b_run.value
        finite = np.concatenate([d['raw_pace'], to_pace(batch)])
        finite = finite[np.isfinite(finite)]
        if finite.size:
            lo, hi = np.nanpercentile(finite, [2, 98])
            pad = 0.15 * (hi - lo) + 0.1
            bfig.layout.yaxis.update(range=[hi + pad, lo - pad], autorange=False)
    rec = float(np.nanmax(r[S.DISTANCE_M].to_numpy()))
    cal = 100.0 * (float(integrate_speed(batch, dt)[-1]) - rec) / rec if np.isfinite(rec) and rec > 0 else float('nan')
    b_metrics.value = (
        f"<b>lag</b> {lag:.1f} s{' (estimated)' if b_auto.value else ''} &nbsp;|&nbsp; "
        f"<b>calibration</b> {cal:+.2f}% (segments {d['cal_pct']:+.2f}%) &nbsp;|&nbsp; "
        f"<b>speed 2nd-diff std</b> {np.std(np.diff(batch, 2)):.4f} m/s"
    )


b_run.observe(b_update, names='value')
b_auto.observe(b_update, names='value')
for _w in b_sliders.values():
    _w.observe(b_update, names='value')
b_update()
display(widgets.VBox([b_metrics, widgets.HBox([widgets.VBox([b_run, b_auto, *b_sliders.values()]), bfig])]))

## Experiment: two-stage fit (segment polynomials -> global fit)

A research sandbox. **Stage 1** fits each cadence segment with a polynomial of adjustable degree (`segment DoF`); **stage 2** samples that skeleton at equidistant control points (`sample dist`); **stage 3** ties the points into one curve with a B-spline of adjustable degree (`global DoF`). Higher degrees add curvature but also let the curve overshoot or chase GNSS noise; `pace 2nd-diff std` (lower = smoother) and the visible control points help find the sweet spot.

The **cadence overlay** checkbox multiplies each steady plateau by normalized cadence for a natural-looking texture. It is **visualization only, never the training target**: cadence is the model's own input, so shaping the target with it would leak.

In [ ]:
from scipy.interpolate import make_interp_spline
from scipy.ndimage import median_filter
from scipy.signal import savgol_filter

from pace_predict.groundtruth import calibrate_ground_truth, lap_segments, median_clean


def seg_poly(base, segs, seg_deg):
    """Stage 1: fit each segment with a degree-seg_deg polynomial (0 => constant median)."""
    skel = np.empty(len(base))
    for a, b, _phase in segs:
        block = base[a:b]
        degree = min(int(seg_deg), len(block) - 1)
        if degree < 1:
            skel[a:b] = float(np.median(block))
        else:
            x = np.arange(len(block), dtype=float)
            skel[a:b] = np.polyval(np.polyfit(x, block, degree), x)
    return skel


def global_fit(skel, segs, sample_s, global_deg, dt):
    """Stages 2-3: sample the skeleton + segment boundaries, B-spline (degree global_deg) through them."""
    n = len(skel)
    step = max(1, int(round(sample_s / dt)))
    idx = set(range(0, n, step)) | {0, n - 1}
    for a, b, _phase in segs:
        idx.add(a)
        idx.add(min(b, n - 1))
    xs = np.array(sorted(idx))
    k = int(min(global_deg, len(xs) - 1))
    curve = make_interp_spline(xs, skel[xs], k=max(k, 1))(np.arange(n))
    return np.clip(curve, 0.0, None), xs


def cadence_texture(curve, segs, cadence):
    """Viz only: modulate each steady plateau by normalized cadence (v ~ stride x cadence)."""
    out = curve.copy()
    fill = float(np.nanmedian(cadence)) if np.isfinite(cadence).any() else 0.0
    cad = median_filter(np.nan_to_num(cadence, nan=fill), size=5)
    window = min(9, len(cad) if len(cad) % 2 else len(cad) - 1)
    if window >= 3:  # a quadratic Savitzky-Golay fit needs at least 3 samples
        cad = savgol_filter(cad, window, 2)
    for a, b, phase in segs:
        mean_c = float(np.mean(cad[a:b]))
        if phase == STEADY and mean_c > 0:
            out[a:b] = out[a:b] * (cad[a:b] / mean_c)
    return np.clip(out, 0.0, None)


ex_run = widgets.Dropdown(options=runs, value=default_run, description='run', **S_)
ex_seg = widgets.IntSlider(1, min=0, max=4, description='segment DoF', **S_)
ex_sample = widgets.FloatSlider(6, min=2, max=20, step=1, description='sample dist (s)', **S_)
ex_global = widgets.IntSlider(3, min=1, max=5, description='global DoF', **S_)
ex_cad = widgets.Checkbox(value=False, description='cadence overlay (viz only, leaky)')
ex_metrics = widgets.HTML()

exfig = go.FigureWidget()
exfig.add_trace(go.Scatter(name='raw enhanced_speed', line=dict(color='lightgray', width=1)))
exfig.add_trace(go.Scatter(name='two-stage fit', line=dict(color='royalblue', width=2.2)))
exfig.add_trace(go.Scatter(name='control points', mode='markers', marker=dict(color='black', size=4)))
exfig.add_trace(go.Scatter(name='cadence overlay', line=dict(color='seagreen', width=1.6)))
exfig.update_layout(height=460, margin=dict(t=30, b=40), xaxis=dict(title='time (min)'),
                    yaxis=dict(title='pace (min/km)'), legend=dict(orientation='h', y=1.1))


def ex_update(_=None):
    act = load_activity(DATA / ex_run.value)
    r = act.records
    dt = median_dt_s(r)
    mnt = r[S.T_S].to_numpy() / 60.0
    speed = r[S.SPEED_MPS].to_numpy()
    cadence = r[S.CADENCE_SPM].to_numpy()
    segs = phase_segments(phase_labels(cadence, dt))
    skel = seg_poly(median_clean(speed, dt, 7.0), segs, ex_seg.value)
    curve, xs = global_fit(skel, segs, ex_sample.value, ex_global.value, dt)
    gt = calibrate_ground_truth(curve, dt, lap_segments(r, act.laps))
    rec = float(np.nanmax(r[S.DISTANCE_M].to_numpy()))
    cal_pct = 100.0 * (float(integrate_speed(gt, dt)[-1]) - rec) / rec if np.isfinite(rec) and rec > 0 else float('nan')
    with exfig.batch_update():
        exfig.data[0].x, exfig.data[0].y = mnt, to_pace(speed)
        exfig.data[1].x, exfig.data[1].y = mnt, to_pace(gt)
        exfig.data[2].x, exfig.data[2].y = mnt[xs], to_pace(gt[xs])
        if ex_cad.value:
            exfig.data[3].x, exfig.data[3].y = mnt, to_pace(cadence_texture(gt, segs, cadence))
        else:
            exfig.data[3].x, exfig.data[3].y = [], []
        exfig.layout.title.text = ex_run.value
        finite = np.concatenate([to_pace(speed), to_pace(gt)])
        finite = finite[np.isfinite(finite)]
        if finite.size:
            lo, hi = np.nanpercentile(finite, [2, 98])
            pad = 0.15 * (hi - lo) + 0.1
            exfig.layout.yaxis.update(range=[hi + pad, lo - pad], autorange=False)
    ex_metrics.value = (
        f"<b>calibration</b> {cal_pct:+.2f}% &nbsp;|&nbsp; <b>control points</b> {len(xs)} &nbsp;|&nbsp; "
        f"<b>pace 2nd-diff std</b> {float(np.nanstd(np.diff(to_pace(gt), 2))):.3f} (lower = smoother)"
    )


for _w in (ex_run, ex_seg, ex_sample, ex_global, ex_cad):
    _w.observe(ex_update, names='value')
ex_update()
display(widgets.VBox([ex_metrics, widgets.HBox([widgets.VBox([ex_run, ex_seg, ex_sample, ex_global, ex_cad]), exfig])]))

## Per-lap validation
Over ≥1 km, GNSS pace is trustworthy, so the exact lap pace (distance/time) and `enhanced_speed` agree to a few s/km. Short reps diverge (GNSS error) — and cadence/GCT still separate work from recovery, which is what the model will exploit.

In [ ]:
def per_lap_table(act) -> pd.DataFrame:
    r = act.records
    t_s = r[S.T_S].to_numpy()
    start0 = r[S.TIMESTAMP].min()
    speed = r[S.SPEED_MPS].to_numpy()
    cadence = r[S.CADENCE_SPM].to_numpy()
    gct = r[S.GROUND_CONTACT_TIME_MS].to_numpy()

    def mean_int(values):  # None when a lap has no samples (e.g. GCT on a multisport bike leg)
        return None if not np.isfinite(values).any() else round(float(np.nanmean(values)))

    ends = [(e - start0).total_seconds() if not pd.isna(e) else float(t_s[-1])
            for e in act.laps[S.LAP_END_TIME]]
    starts = [0.0, *ends[:-1]]
    rows = []
    for i, (a, b) in enumerate(zip(starts, ends, strict=True)):
        mask = (t_s >= a) & (t_s < b)
        distance = act.laps[S.LAP_DISTANCE_M].iloc[i]
        duration = act.laps[S.LAP_ELAPSED_S].iloc[i]
        if mask.sum() == 0 or not (np.isfinite(distance) and distance > 0):
            continue
        if not (np.isfinite(duration) and duration > 0):
            continue
        rows.append({
            'lap': i, 'dist_m': round(distance),
            'exact_pace': round(1000.0 / (distance / duration) / 60.0, 2),
            'es_pace': round(float(np.nanmean(to_pace(speed[mask]))), 2),
            'cadence': mean_int(cadence[mask]),
            'gct_ms': mean_int(gct[mask]),
            'trusted_>=1km': distance >= 1000,
        })
    return pd.DataFrame(rows)


per_lap_table(load_activity(DATA / run_dd.value))